# EVASPA Evapotranspiration processing

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime as dt
import os
from pprint import pprint

import numpy as np
import xarray as xr

In [ ]:
from sweat.api import filter_data_for_evaspa
from sweat.common import flux, io
from sweat.evaspa import ef, merging, seb
from sweat.evaspa.config import EVASPAParamsConfig

In [ ]:
import sweat.extra.plot_notebook as pltn

### Input

Read data

In [ ]:
src_path = os.path.dirname(os.path.dirname(os.getcwd()))
data = io.read_data_from_file(
    os.path.join(src_path, "tests", "data", "data_test_20230303T112730.tif")
)
# Add date to metadata
data.attrs["date"] = dt.datetime(2018, 5, 16, 10, 0, 0, tzinfo=dt.UTC)

In [ ]:
sorted(data.data_vars)

In [ ]:
data

Generate cloud mask (for demonstration purpose)

In [ ]:
cloud_mask = xr.ones_like(data["lst"])
data["cloud"] = cloud_mask.where(
    (cloud_mask.x >= -16.2)
    & (cloud_mask.x <= -16)
    & (cloud_mask.y >= 14.7)
    & (cloud_mask.y <= 14.8),
    0,
)

Generate albedo invalid data (for demonstration purpose)

In [ ]:
data["albedo"] = data["albedo"].where(
    ~(
        (data["albedo"].x >= -16.85)
        & (data["albedo"].x <= -16.8)
        & (data["albedo"].y >= 14.4)
        & (data["albedo"].y <= 14.45)
    ),
    np.nan,
)

Plot data

In [ ]:
pltn.plot_band(data, band="lst")

In [ ]:
pltn.plot_band(data, band="cloud")

In [ ]:
pltn.plot_band(data, band="albedo")

In [ ]:
pltn.plot_data(data, "albedo")

Configuration

In [ ]:
params = {
    "filtering": {"cloud": {"op": "!=", "value": 1}},
    "ef": {
        "check": {"threshold": 0.02},
        "models": "default_evaspa",
        "options": {
            "filtering": {
                "albedo": {
                    "and": [
                        {"op": ">=", "value": 0.1},
                        {"op": "<=", "value": 0.3},
                    ]
                }
            },
            "selection": False,
            "merging": {
                "merging_method": "median",
                "uncertainty_method": "interquartile",
            },
        },
    },
    "seb": {"use_topo": False, "models": ["kustas"]},
}

## Run step by step 

Validate parameters config

In [ ]:
# Validate parameters config
params_config = EVASPAParamsConfig.model_validate(params)

In [ ]:
params_config

In [ ]:
pprint(params_config.model_dump(by_alias=True), sort_dicts=False)

Initalize

In [ ]:
models, options = ef.initialize(params_config.ef.model_dump(by_alias=True))

Filter

  - Identify valid pixels for edge computation
  - Identify invalid pixels

In [ ]:
data = filter_data_for_evaspa(
    data=data, models=models, config=params_config.filtering.model_dump()
)

Plot valid mask

In [ ]:
if (data["valid"].data == 0).sum() == 0:
    print("No invalid pixels")
else:
    pltn.plot_mask(data=data, mask="valid", invert=True)

Plot flags

In [ ]:
data["flags"].plot()

Check variability

In [ ]:
ef.check_variability(
    data["lst"], mask=data["valid"], **params_config.ef.check.model_dump()
)

Compute EF

In [ ]:
models

In [ ]:
ef_xr, ef_inst_xr = ef.run(
    models, data, **params_config.ef.options.model_dump(by_alias=True)
)

In [ ]:
ef_xr.data_vars

In [ ]:
ef_xr

In [ ]:
pltn.plot_models(data, ef_xr)

In [ ]:
pltn.plot_dataset(ef_xr)

Compute LE

In [ ]:
print(f"Constant Stefan Boltzmann = {flux.CST_SB}")

Compute net radiation

In [ ]:
rn_xr = seb.create_net_radiation(data, use_topo=True)

In [ ]:
rn_xr

In [ ]:
pltn.plot_dataset(rn_xr)

Compute G/Rn ratio

In [ ]:
ratio_xr = seb.create_ratio(data, models=params_config.seb.models)

In [ ]:
ratio_xr

In [ ]:
pltn.plot_dataset(ratio_xr)

In [ ]:
le_xr = seb.create_le(ef_xr, rn_xr, ratio_xr)

In [ ]:
le_xr

In [ ]:
pltn.plot_dataset(le_xr)

Merge LE

In [ ]:
le_inst = merging.merge_to_dataset(
    le_xr,
    merging_method=merging.MergingMethod.MEDIAN,
    uncertainty_method=merging.UncertaintyMethod.INTERQUARTILE,
    name="le",
)

In [ ]:
le_inst

Visualize results

In [ ]:
pltn.plot_dataset(le_inst)